# Apache Spark Koalas

Este notebook contiene las funciones principales de Koalas, obtenidas de la documentación oficial de https://koalas.readthedocs.io/

In [4]:
#!pip install plotly

In [5]:
import sys
sys.executable

'/opt/homebrew/anaconda3/envs/ProcesBigData/bin/python'

### Importar librerias

In [6]:
import findspark
findspark.init()

import pyspark
import numpy as np
import pandas as pd

from pyspark.sql import SparkSession

In [7]:
import pyspark.pandas as ks

### 1. Creación de objetos



Creando una serie Koalas pasando una lista de valores, permitiendo que Koalas cree un índice entero predeterminado:

In [8]:
s = ks.Series([1, 3, 5, np.nan, 6, 8])

In [9]:
s

0    1.0
1    3.0
2    5.0
3    NaN
4    6.0
5    8.0
dtype: float64

Creando un Koalas DataFrame pasando un dict de objetos que se pueden convertir a series.

In [10]:
kdf = ks.DataFrame(
    {'a': [1, 2, 3, 4, 5, 6],
     'b': [100, 200, 300, 400, 500, 600],
     'c': ["one", "two", "three", "four", "five", "six"]},
    index=[10, 20, 30, 40, 50, 60])

### Muestra el dataframe de koalas

In [11]:
kdf

,a,b,c
10,1,100,one
20,2,200,two
30,3,300,three
40,4,400,four
50,5,500,five
60,6,600,six


Creando un DataFrame de pandas pasando una matriz numpy, con un índice de fecha y hora y columnas etiquetadas:

### Usando pandas se crea un dataframe que va a crear un rango de fechas a partir de la primera fecha que se le paso y seran 6 fechas ya que se determino que serian 6 periodos.

In [12]:
dates = pd.date_range('20130101', periods=6)

### Para ver que fechas se generaron en el dataframe

In [13]:
dates

DatetimeIndex(['2013-01-01', '2013-01-02', '2013-01-03', '2013-01-04',
               '2013-01-05', '2013-01-06'],
              dtype='datetime64[ns]', freq='D')

### Se crea un DataFrame de pandas llamado pdf con 6 filas, 4 columnas, y números aleatorios. El índice es una serie de fechas (antes creadas), y las columnas están etiquetadas como 'A', 'B', 'C', y 'D'

In [14]:
pdf = pd.DataFrame(np.random.randn(6, 4), index=dates, columns=list('ABCD'))

### Se esta mostrando el dataframe

In [15]:
pdf

,A,B,C,D
2013-01-01,1.236089,0.951097,-1.538436,-2.600304
2013-01-02,0.817790,0.207778,-2.383624,0.122441
2013-01-03,0.311807,1.463852,-1.095609,0.315084
2013-01-04,0.765981,0.650983,-0.400122,0.194215
2013-01-05,1.054743,1.865647,-0.238439,-0.552411
2013-01-06,-1.072626,-0.385433,0.672013,-0.860601


Ahora, este DataFrame de pandas se puede convertir en un DataFrame de Koalas

### Se esta convirtiendo un Dataframe de pandas a koalas con la funcion ks.from_pandas()


In [16]:
kdf = ks.from_pandas(pdf)

### Para conocer el tipo de variable

In [17]:
type(kdf)

pyspark.pandas.frame.DataFrame

Sin embargo, se ve y se comporta igual que un DataFrame de pandas

### Mostrando como se ve el Dataframe de koalas para compararlo visualmente contra el dataframe de pandas(unas celdas antes)

In [18]:
kdf

,A,B,C,D
2013-01-01,1.236089,0.951097,-1.538436,-2.600304
2013-01-02,0.817790,0.207778,-2.383624,0.122441
2013-01-03,0.311807,1.463852,-1.095609,0.315084
2013-01-04,0.765981,0.650983,-0.400122,0.194215
2013-01-05,1.054743,1.865647,-0.238439,-0.552411
2013-01-06,-1.072626,-0.385433,0.672013,-0.860601


Además, es posible crear un **Koalas DataFrame desde Spark DataFrame**.

Creando un Spark DataFrame a partir de pandas DataFrame

### Se está creando o obteniendo una instancia de SparkSession, que es el punto de entrada principal para trabajar con Spark en PySpark

In [19]:
spark = SparkSession.builder.getOrCreate()

### Convierte el DataFrame de pandas llamado pdf a un DataFrame de Spark, asignándolo a la variable sdf

In [20]:
sdf = spark.createDataFrame(pdf)

### Para mostrar un dateframe de spark

In [21]:
sdf.show()

+-------------------+--------------------+--------------------+-------------------+
|                  A|                   B|                   C|                  D|
+-------------------+--------------------+--------------------+-------------------+
| 1.2360885095964902|  0.9510967563335281| -1.5384358386941592| -2.600303925028371|
| 0.8177896789493494|  0.2077781683206595| -2.3836238296544914|0.12244121706636782|
| 0.3118073412933641|  1.4638522256940252| -1.0956093470408612|  0.315083939617673|
| 0.7659814781266149|  0.6509826501210958| -0.4001221543131267|0.19421457024171748|
| 1.0547434134273614|  1.8656466985580662|-0.23843899313670291|-0.5524108728070629|
|-1.0726260355118733|-0.38543290901777383|  0.6720130992201243|-0.8606009592197355|
+-------------------+--------------------+--------------------+-------------------+



Creando Koalas DataFrame desde Spark DataFrame.
`to_koalas ()` se adjunta automáticamente a Spark DataFrame y está disponible como una API cuando se importa Koalas.

### convierte un DataFrame de Spark (sdf) a un DataFrame de Koalas 

In [22]:
kdf = sdf.to_koalas()

In [23]:
kdf

,A,B,C,D
0,1.236089,0.951097,-1.538436,-2.600304
1,0.817790,0.207778,-2.383624,0.122441
2,0.311807,1.463852,-1.095609,0.315084
3,0.765981,0.650983,-0.400122,0.194215
4,1.054743,1.865647,-0.238439,-0.552411
5,-1.072626,-0.385433,0.672013,-0.860601


Tiene [dtypes] específicos. Actualmente se admiten los tipos que son comunes a Spark y pandas.

### Permitirá ver los tipos de datos de cada columna en el DataFrame kdf

In [24]:
kdf.dtypes

A    float64
B    float64
C    float64
D    float64
dtype: object

### 2. Manipulación de datos


A diferencia de los pandas, los datos en un dataframe de datos de Spark no están _ordenados_, no tienen una noción intrínseca de índice. Cuando se le solicite el encabezado, Spark solo tomará el número solicitado de filas de una partición. **No hay que utilizar el df de Koalas para devolver filas específicas**, use `.loc` o` iloc` en su lugar.

### permite ver 5 filas del DataFrame (no necesariamente regresa las primeras lineas)

In [25]:
kdf.head()

,A,B,C,D
0,1.236089,0.951097,-1.538436,-2.600304
1,0.817790,0.207778,-2.383624,0.122441
2,0.311807,1.463852,-1.095609,0.315084
3,0.765981,0.650983,-0.400122,0.194215
4,1.054743,1.865647,-0.238439,-0.552411


Muestre el índice, las columnas y los datos numéricos subyacentes.

También puede recuperar el índice; la columna de índice se puede atribuir a un DataFrame, ver más adelante

In [26]:
kdf.index

Int64Index([0, 1, 2, 3, 4, 5], dtype='int64')

### Para que se regrese el nombre de las columnas

In [27]:
kdf.columns

Index(['A', 'B', 'C', 'D'], dtype='object')

### Se esta convirtiendo el Dataframe a un arreglo.


In [28]:
kdf.to_numpy()

/opt/homebrew/Cellar/apache-spark/3.5.3/libexec/python/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_numpy` loads all data into the driver's memory. It should only be used if the resulting NumPy ndarray is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


array([[ 1.23608851,  0.95109676, -1.53843584, -2.60030393],
       [ 0.81778968,  0.20777817, -2.38362383,  0.12244122],
       [ 0.31180734,  1.46385223, -1.09560935,  0.31508394],
       [ 0.76598148,  0.65098265, -0.40012215,  0.19421457],
       [ 1.05474341,  1.8656467 , -0.23843899, -0.55241087],
       [-1.07262604, -0.38543291,  0.6720131 , -0.86060096]])

**Describe** muestra un resumen estadístico rápido de sus datos

In [29]:
kdf.describe()

24/10/28 13:14:21 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


,A,B,C,D
count,6.000000,6.000000,6.000000,6.000000
mean,0.518964,0.792321,-0.830703,-0.563596
std,0.839966,0.821823,1.074891,1.100320
min,-1.072626,-0.385433,-2.383624,-2.600304
25%,0.311807,0.207778,-1.538436,-0.860601
50%,0.765981,0.650983,-1.095609,-0.552411
75%,1.054743,1.463852,-0.238439,0.194215
max,1.236089,1.865647,0.672013,0.315084


Transposición de sus datos

### Para trasponer la informacion (cambiar filas por columnas y viceversa)

In [30]:
kdf.T

,0,1,2,3,4,5
A,1.236089,0.817790,0.311807,0.765981,1.054743,-1.072626
B,0.951097,0.207778,1.463852,0.650983,1.865647,-0.385433
C,-1.538436,-2.383624,-1.095609,-0.400122,-0.238439,0.672013
D,-2.600304,0.122441,0.315084,0.194215,-0.552411,-0.860601


Ordenando por su índice

### .sort_index por default ordena del primero al ultimo, al agregar el False lo ordena al reves


In [31]:
kdf.sort_index(ascending=False)

,A,B,C,D
5,-1.072626,-0.385433,0.672013,-0.860601
4,1.054743,1.865647,-0.238439,-0.552411
3,0.765981,0.650983,-0.400122,0.194215
2,0.311807,1.463852,-1.095609,0.315084
1,0.817790,0.207778,-2.383624,0.122441
0,1.236089,0.951097,-1.538436,-2.600304


Ordenar por valor

### .sort_values ordena los valores, en este caso se esta ordenando tomando en cuenta los valores de la columna B

In [32]:
kdf.sort_values(by='B')

,A,B,C,D
5,-1.072626,-0.385433,0.672013,-0.860601
1,0.817790,0.207778,-2.383624,0.122441
3,0.765981,0.650983,-0.400122,0.194215
0,1.236089,0.951097,-1.538436,-2.600304
2,0.311807,1.463852,-1.095609,0.315084
4,1.054743,1.865647,-0.238439,-0.552411


### 3. Datos faltantes
Koalas utiliza principalmente el valor `np.nan` para representar los datos faltantes. Por defecto, no se incluye en los cálculos.


### Se está tomando un DataFrame de pandas y lo está reorganizando o redimensionando, se crea un nuevo DataFrame pdf1 con las primeras cuatro fechas como filas y con las columnas originales de pdf más una columna adicional llamada 'E', que tendrá valores NaN en todos los registros iniciales

In [33]:
pdf1 = pdf.reindex(index=dates[0:4], columns=list(pdf.columns) + ['E'])
pdf1

,A,B,C,D,E
2013-01-01,1.236089,0.951097,-1.538436,-2.600304,NaN
2013-01-02,0.817790,0.207778,-2.383624,0.122441,NaN
2013-01-03,0.311807,1.463852,-1.095609,0.315084,NaN
2013-01-04,0.765981,0.650983,-0.400122,0.194215,NaN


### Se está asignando el valor 1 a la columna 'E' del dataframe, pero solo para las filas correspondientes a las fechas entre dates[0] y dates[1]

In [34]:
pdf1.loc[dates[0]:dates[1], 'E'] = 1
pdf1

,A,B,C,D,E
2013-01-01,1.236089,0.951097,-1.538436,-2.600304,1.0
2013-01-02,0.817790,0.207778,-2.383624,0.122441,1.0
2013-01-03,0.311807,1.463852,-1.095609,0.315084,NaN
2013-01-04,0.765981,0.650983,-0.400122,0.194215,NaN


### está convirtiendo el DataFrame de pandas pdf1 en un DataFrame de Koalas y asignándolo a kdf1


In [35]:
kdf1 = ks.from_pandas(pdf1)

In [36]:
kdf1

,A,B,C,D,E
2013-01-01,1.236089,0.951097,-1.538436,-2.600304,1.0
2013-01-02,0.817790,0.207778,-2.383624,0.122441,1.0
2013-01-03,0.311807,1.463852,-1.095609,0.315084,NaN
2013-01-04,0.765981,0.650983,-0.400122,0.194215,NaN


Para eliminar las filas que tienen datos faltantes.

### eliminando de kdf1 todas las filas que contengan al menos un valor NaN

In [37]:
kdf1.dropna(how='any')

,A,B,C,D,E
2013-01-01,1.236089,0.951097,-1.538436,-2.600304,1.0
2013-01-02,0.817790,0.207778,-2.383624,0.122441,1.0


Llenando los datos faltantes.

### De lo que no tenga valor se va a llenar con valor de 5 en esas celdas

In [38]:
kdf1.fillna(value=5)

,A,B,C,D,E
2013-01-01,1.236089,0.951097,-1.538436,-2.600304,1.0
2013-01-02,0.817790,0.207778,-2.383624,0.122441,1.0
2013-01-03,0.311807,1.463852,-1.095609,0.315084,5.0
2013-01-04,0.765981,0.650983,-0.400122,0.194215,5.0


### 4. Operaciones

#### Estadísticas
Las operaciones en general excluyen los datos faltantes.

Realización de una estadística descriptiva:

In [39]:
kdf.mean()

A    0.518964
B    0.792321
C   -0.830703
D   -0.563596
dtype: float64

#### Configuraciones de Spark

Varias configuraciones en PySpark se pueden aplicar internamente en Koalas.
Por ejemplo, puede habilitar la optimización de Arrow para acelerar enormemente la conversión de pandas internos.

### prev obtiene el valor actual de la configuración de Spark que habilita las optimizaciones de Arrow para la ejecución de consultas. Se guarda en prev para referenciarla más adelante
### ks.set_option establece el tipo de índice por defecto en Koalas como "distribuido".
### y luego se ignoran los warnings 

In [40]:
prev = spark.conf.get("spark.sql.execution.arrow.enabled")  # Keep its default value.
ks.set_option("compute.default_index_type", "distributed")  # Use default index prevent overhead.

import warnings
warnings.filterwarnings("ignore")  # Ignore warnings coming from Arrow optimizations.

In [41]:
spark.conf.set("spark.sql.execution.arrow.enabled", True)
%timeit ks.range(300000).to_pandas()

24/10/28 13:14:23 WARN SQLConf: The SQL config 'spark.sql.execution.arrow.enabled' has been deprecated in Spark v3.0 and may be removed in the future. Use 'spark.sql.execution.arrow.pyspark.enabled' instead of it.


82.3 ms ± 30.8 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [42]:
spark.conf.set("spark.sql.execution.arrow.enabled", False)
%timeit ks.range(300000).to_pandas()

24/10/28 13:14:24 WARN SQLConf: The SQL config 'spark.sql.execution.arrow.enabled' has been deprecated in Spark v3.0 and may be removed in the future. Use 'spark.sql.execution.arrow.pyspark.enabled' instead of it.


362 ms ± 15.6 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [43]:
ks.reset_option("compute.default_index_type")
spark.conf.set("spark.sql.execution.arrow.enabled", prev)  # Set its default value back.

24/10/28 13:14:27 WARN SQLConf: The SQL config 'spark.sql.execution.arrow.enabled' has been deprecated in Spark v3.0 and may be removed in the future. Use 'spark.sql.execution.arrow.pyspark.enabled' instead of it.


#### Agrupación
Por "agrupar por" nos referimos a un proceso que involucra uno o más de los siguientes pasos:

- Dividir los datos en grupos según algunos criterios.
- Aplicar una función a cada grupo de forma independiente
- Combinar los resultados en una estructura de datos

### Se crea un DataFrame de Koalas (kdf) con cuatro columnas, cada una con diferentes tipos de datos.

In [44]:
kdf = ks.DataFrame({'A': ['foo', 'bar', 'foo', 'bar',
                          'foo', 'bar', 'foo', 'foo'],
                    'B': ['one', 'one', 'two', 'three',
                          'two', 'two', 'one', 'three'],
                    'C': np.random.randn(8),
                    'D': np.random.randn(8)})

### Para ense;ar el datagframe

In [45]:
kdf

,A,B,C,D
0,foo,one,0.199519,0.944810
1,bar,one,1.029303,0.467135
2,foo,two,0.377687,-0.579014
3,bar,three,0.299227,-0.778003
4,foo,two,-0.966238,-0.245831
5,bar,two,-0.898678,-1.016474
6,foo,one,-0.275946,-0.858468
7,foo,three,-0.117726,0.401186


Agrupar y luego aplicar el **sum** a los grupos resultantes.

### Se esta agrupando la informacion que hay basandose en la columna A y se suma toda la informacion que pertenece al mismo grupo

In [46]:
kdf.groupby('A').sum()

,C,D
A,,
foo,-0.782704,-0.337318
bar,0.429852,-1.327342


In [47]:
kdf.groupby(['A', 'B']).sum()

C         D
A   B                        
foo one   -0.076426  0.086342
bar one    1.029303  0.467135
foo two   -0.588552 -0.824846
bar three  0.299227 -0.778003
    two   -0.898678 -1.016474
foo three -0.117726  0.401186

### 5. Visualización de datos: Generar gráficos


### Creando una serie de pandas llamada pser que contiene 1000 valores aleatorios generados a partir de una distribución normal, tambien se crea un índice de fechas que comienza el 1 de enero de 2000 y tiene una duración de 1000 períodos (días consecutivos). Esto genera un rango de fechas que se utilizará como índice para la serie

In [48]:
pser = pd.Series(np.random.randn(1000),
                 index=pd.date_range('1/1/2000', periods=1000))
pser

2000-01-01    0.043554
2000-01-02    0.697340
2000-01-03   -0.439674
2000-01-04    1.115731
2000-01-05   -0.120596
                ...   
2002-09-22   -2.325921
2002-09-23    0.979366
2002-09-24   -0.844767
2002-09-25    0.140777
2002-09-26   -0.513989
Freq: D, Length: 1000, dtype: float64

### Creando una nueva serie de Koalas a partir de la serie de pandas llamada pser

In [49]:
kser = ks.Series(pser)

### Devuelve el valor máximo de todos los elementos anteriores hasta ese punto

In [50]:
kser = kser.cummax()
kser

24/10/28 13:14:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
24/10/28 13:14:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
24/10/28 13:14:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
24/10/28 13:14:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
24/10/28 13:14:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


2000-01-01    0.043554
2000-01-02    0.697340
2000-01-03    0.697340
2000-01-04    1.115731
2000-01-05    1.115731
2000-01-06    1.115731
2000-01-07    1.115731
2000-01-08    1.115731
2000-01-09    1.115731
2000-01-10    1.115731
2000-01-11    1.115731
2000-01-12    1.115731
2000-01-13    1.115731
2000-01-14    1.115731
2000-01-15    1.115731
2000-01-16    1.115731
2000-01-17    1.115731
2000-01-18    1.115731
2000-01-19    1.115731
2000-01-20    1.115731
2000-01-21    1.115731
2000-01-22    1.115731
2000-01-23    1.115731
2000-01-24    1.115731
2000-01-25    1.115731
2000-01-26    1.552059
2000-01-27    1.552059
2000-01-28    1.552059
2000-01-29    1.552059
2000-01-30    1.552059
2000-01-31    1.552059
2000-02-01    1.552059
2000-02-02    1.552059
2000-02-03    1.552059
2000-02-04    1.552059
2000-02-05    1.552059
2000-02-06    1.552059
2000-02-07    1.552059
2000-02-08    1.552059
2000-02-09    1.552059
2000-02-10    1.552059
2000-02-11    1.552059
2000-02-12    1.552059
2000-02-13 

**Gráficos de matplotlib**

### Generar un grafico de lineas con la libreria de matplotlib 

In [51]:
ks.options.plotting.backend = "matplotlib"

kser.plot()

ImportError: matplotlib is required for plotting when the default backend 'matplotlib' is selected.

**Gráficos de Pandas Bokeh**

### Se genera un grafico con bokeh, hacer un gráfico interactivo que muestra los valores acumulativos a lo largo del tiempo, donde puedes hacer zoom, desplazarte y obtener más información al pasar el mouse sobre los puntos del gráfico.

In [ ]:
ks.options.plotting.backend = "pandas_bokeh"

kdf.plot(backend="pandas_bokeh", title="Example Figure")
kser.plot()


**Gráfico por defecto con plotly**

In [ ]:
kser.plot()

En un DataFrame, el plot() es una conveniencia para trazar todas las columnas con etiquetas:

In [45]:
pdf = pd.DataFrame(np.random.randn(1000, 4), index=pser.index,
                   columns=['A', 'B', 'C', 'D'])

In [71]:
kdf = ks.from_pandas(pdf)

### Devuelve el valor máximo de todos los elementos anteriores hasta ese punto

In [ ]:
kdf = kdf.cummax()
kdf

In [ ]:
kdf.plot()

### 6. Entrada / salida de datos


### CSV

CSV es sencillo y fácil de usar

### Convertir el dataframer en csv y luego leer las primeras 10 lineas

In [ ]:
kdf.to_csv('foo.csv')
ks.read_csv('foo.csv').head(10)

### Parquet

Parquet es un formato de archivo eficiente y compacto para leer y escribir más rápido.

### se convierte el archivo en parquet que es un formato de almacenamiento de datos,  permite recuperar los datos de manera eficiente

In [ ]:
kdf.to_parquet('bar.parquet')
ks.read_parquet('bar.parquet').head(10)

### Spark IO

Además, Koalas es totalmente compatible con las diversas fuentes de datos de Spark, como ORC y una fuente de datos externa.

### guardar un DataFrame de Koalas (kdf) en un formato ORC (Optimized Row Columnar) y luego leer ese archivo ORC en un DataFrame de Koalas

In [ ]:
kdf.to_spark_io('zoo.orc', format="orc")
ks.read_spark_io('zoo.orc', format="orc").head(10)